# Smart Energy Consumption Analysis using Smart Meter Data

---

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from scipy.stats import chi2_contingency, ttest_ind
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
print('All libraries imported successfully.')

## Section 2: Data Loading

In [ ]:
df = pd.read_csv('data/daily_dataset.csv')
print('Dataset Shape:', df.shape)
print('\nFirst 5 Rows:')
df.head()

In [ ]:
print('Column Names:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print('\nDataset Info:')
df.info()

## Section 3: Data Cleaning

In [ ]:
print('Missing Values Per Column:')
print(df.isnull().sum())
print('\nTotal Missing Values:', df.isnull().sum().sum())
print('Missing Value Percentage:')
print(round(df.isnull().mean() * 100, 2))

In [ ]:
df_clean = df.copy()

numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
for col in numeric_cols:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

cat_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)

print('Missing Values After Cleaning:')
print(df_clean.isnull().sum())

In [ ]:
before = df_clean.shape[0]
df_clean.drop_duplicates(inplace=True)
after = df_clean.shape[0]
print(f'Duplicate rows removed: {before - after}')
print(f'Dataset size after cleaning: {df_clean.shape}')

## Section 4: Data Wrangling

In [ ]:
date_col = None
for col in df_clean.columns:
    if 'date' in col.lower() or 'day' in col.lower() or 'time' in col.lower():
        date_col = col
        break

if date_col:
    df_clean[date_col] = pd.to_datetime(df_clean[date_col], errors='coerce')
    df_clean['Year']    = df_clean[date_col].dt.year
    df_clean['Month']   = df_clean[date_col].dt.month
    df_clean['Day']     = df_clean[date_col].dt.day
    df_clean['DayOfWeek'] = df_clean[date_col].dt.dayofweek
    df_clean['WeekNumber'] = df_clean[date_col].dt.isocalendar().week.astype(int)
    df_clean['Quarter'] = df_clean[date_col].dt.quarter
    df_clean['IsWeekend'] = df_clean['DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)
    print(f'Date column "{date_col}" parsed successfully.')
    print(df_clean[[date_col, 'Year', 'Month', 'Day', 'DayOfWeek', 'IsWeekend']].head())
else:
    print('No date column found. Skipping date feature extraction.')

In [ ]:
energy_col = None
for col in numeric_cols:
    if any(k in col.lower() for k in ['energy','kwh','consumption','usage','sum','mean']):
        energy_col = col
        break

if energy_col is None and len(numeric_cols) > 0:
    energy_col = numeric_cols[0]

print(f'Primary energy column identified: "{energy_col}"')

df_clean['Consumption_KWh'] = df_clean[energy_col]

mean_val = df_clean['Consumption_KWh'].mean()
df_clean['Usage_Category'] = pd.cut(
    df_clean['Consumption_KWh'],
    bins=[df_clean['Consumption_KWh'].min()-0.001,
          mean_val * 0.5,
          mean_val * 1.2,
          df_clean['Consumption_KWh'].max()],
    labels=['Low', 'Medium', 'High']
)
print('\nUsage Category Distribution:')
print(df_clean['Usage_Category'].value_counts())

In [ ]:
if 'Month' in df_clean.columns:
    def get_season(m):
        if m in [12, 1, 2]:  return 'Winter'
        elif m in [3, 4, 5]: return 'Spring'
        elif m in [6, 7, 8]: return 'Summer'
        else:                return 'Autumn'

    df_clean['Season'] = df_clean['Month'].apply(get_season)
    print('Season Distribution:')
    print(df_clean['Season'].value_counts())

## Section 5: Data Transformation

In [ ]:
df_clean['Consumption_Normalized'] = (
    (df_clean['Consumption_KWh'] - df_clean['Consumption_KWh'].min()) /
    (df_clean['Consumption_KWh'].max() - df_clean['Consumption_KWh'].min())
)

df_clean['Consumption_Standardized'] = (
    (df_clean['Consumption_KWh'] - df_clean['Consumption_KWh'].mean()) /
    df_clean['Consumption_KWh'].std()
)

df_clean['Consumption_Log'] = np.log1p(df_clean['Consumption_KWh'])

print('Transformation Preview:')
df_clean[['Consumption_KWh','Consumption_Normalized','Consumption_Standardized','Consumption_Log']].describe()

In [ ]:
if 'Month' in df_clean.columns:
    month_avg = df_clean.groupby('Month')['Consumption_KWh'].transform('mean')
    df_clean['Deviation_From_Monthly_Avg'] = df_clean['Consumption_KWh'] - month_avg
    print('Deviation from Monthly Average — Sample:')
    print(df_clean[['Month','Consumption_KWh','Deviation_From_Monthly_Avg']].head(10))

## Section 6: Exploratory Data Analysis (EDA)

In [ ]:
print('='*60)
print('DESCRIPTIVE STATISTICS')
print('='*60)
print(df_clean[['Consumption_KWh']].describe().round(4))

print(f"\nMean    : {df_clean['Consumption_KWh'].mean():.4f} kWh")
print(f"Median  : {df_clean['Consumption_KWh'].median():.4f} kWh")
print(f"Mode    : {df_clean['Consumption_KWh'].mode()[0]:.4f} kWh")
print(f"Std Dev : {df_clean['Consumption_KWh'].std():.4f} kWh")
print(f"Variance: {df_clean['Consumption_KWh'].var():.4f}")
print(f"Skewness: {df_clean['Consumption_KWh'].skew():.4f}")
print(f"Kurtosis: {df_clean['Consumption_KWh'].kurtosis():.4f}")

### 6.1 Histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['Consumption_KWh'], bins=40, color='steelblue', edgecolor='black', alpha=0.8)
axes[0].axvline(df_clean['Consumption_KWh'].mean(), color='red', linestyle='--', linewidth=2, label=f"Mean: {df_clean['Consumption_KWh'].mean():.2f}")
axes[0].axvline(df_clean['Consumption_KWh'].median(), color='green', linestyle='--', linewidth=2, label=f"Median: {df_clean['Consumption_KWh'].median():.2f}")
axes[0].set_title('Distribution of Daily Energy Consumption')
axes[0].set_xlabel('Energy Consumption (kWh)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].hist(df_clean['Consumption_Log'], bins=40, color='darkorange', edgecolor='black', alpha=0.8)
axes[1].set_title('Log-Transformed Consumption Distribution')
axes[1].set_xlabel('Log(Consumption + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.suptitle('Histogram Analysis', fontsize=15, y=1.02, fontweight='bold')
plt.savefig('histogram_consumption.png', bbox_inches='tight')
plt.show()

In [ ]:
if 'Month' in df_clean.columns:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()
    months = sorted(df_clean['Month'].dropna().unique())[:6]
    month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                   7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

    for i, m in enumerate(months):
        data = df_clean[df_clean['Month'] == m]['Consumption_KWh']
        axes[i].hist(data, bins=25, color=sns.color_palette('tab10')[i], edgecolor='black', alpha=0.8)
        axes[i].axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.2f}')
        axes[i].set_title(f'{month_names.get(m,m)} — Consumption')
        axes[i].set_xlabel('kWh')
        axes[i].set_ylabel('Frequency')
        axes[i].legend(fontsize=8)

    plt.suptitle('Monthly Consumption Histograms', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig('monthly_histograms.png', bbox_inches='tight')
    plt.show()

### 6.2 Box Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bp1 = axes[0].boxplot(df_clean['Consumption_KWh'].dropna(), patch_artist=True,
                       boxprops=dict(facecolor='lightblue', color='navy'),
                       whiskerprops=dict(color='navy'),
                       capprops=dict(color='navy'),
                       medianprops=dict(color='red', linewidth=2),
                       flierprops=dict(marker='o', color='red', alpha=0.5))
axes[0].set_title('Box Plot — Daily Energy Consumption')
axes[0].set_ylabel('Energy Consumption (kWh)')
axes[0].set_xticks([1])
axes[0].set_xticklabels(['All Data'])

if 'Season' in df_clean.columns:
    seasons = ['Winter','Spring','Summer','Autumn']
    season_data = [df_clean[df_clean['Season']==s]['Consumption_KWh'].dropna() for s in seasons]
    season_data = [d for d in season_data if len(d) > 0]
    bp2 = axes[1].boxplot(season_data, patch_artist=True,
                           boxprops=dict(facecolor='lightyellow', color='darkgreen'),
                           whiskerprops=dict(color='darkgreen'),
                           capprops=dict(color='darkgreen'),
                           medianprops=dict(color='red', linewidth=2),
                           flierprops=dict(marker='o', color='red', alpha=0.5))
    axes[1].set_xticks(range(1, len(season_data)+1))
    axes[1].set_xticklabels([s for s in seasons if len(df_clean[df_clean['Season']==s]) > 0], rotation=15)
    axes[1].set_title('Seasonal Consumption Box Plot')
    axes[1].set_ylabel('Energy Consumption (kWh)')
elif 'Month' in df_clean.columns:
    months = sorted(df_clean['Month'].dropna().unique())
    mdata = [df_clean[df_clean['Month']==m]['Consumption_KWh'].dropna() for m in months]
    axes[1].boxplot(mdata, patch_artist=True,
                    medianprops=dict(color='red', linewidth=2),
                    flierprops=dict(marker='.', color='red', alpha=0.3))
    axes[1].set_xticks(range(1, len(months)+1))
    axes[1].set_xticklabels([str(m) for m in months])
    axes[1].set_title('Monthly Consumption Box Plot')
    axes[1].set_ylabel('Energy Consumption (kWh)')
    axes[1].set_xlabel('Month')

plt.tight_layout()
plt.savefig('boxplots.png', bbox_inches='tight')
plt.show()

In [ ]:
if 'IsWeekend' in df_clean.columns:
    plt.figure(figsize=(8, 5))
    df_clean['Day_Type'] = df_clean['IsWeekend'].map({0: 'Weekday', 1: 'Weekend'})
    data_wday  = df_clean[df_clean['IsWeekend']==0]['Consumption_KWh'].dropna()
    data_wkend = df_clean[df_clean['IsWeekend']==1]['Consumption_KWh'].dropna()
    bp = plt.boxplot([data_wday, data_wkend], patch_artist=True,
                     labels=['Weekday', 'Weekend'],
                     boxprops=dict(facecolor='#AED6F1'),
                     medianprops=dict(color='red', linewidth=2),
                     flierprops=dict(marker='o', color='grey', alpha=0.4))
    plt.title('Weekday vs Weekend Energy Consumption')
    plt.ylabel('Energy Consumption (kWh)')
    plt.savefig('weekday_weekend_boxplot.png', bbox_inches='tight')
    plt.show()

## Section 7: IQR Method — Outlier Detection

In [ ]:
Q1  = df_clean['Consumption_KWh'].quantile(0.25)
Q3  = df_clean['Consumption_KWh'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['Consumption_KWh'] < lower_bound) |
                    (df_clean['Consumption_KWh'] > upper_bound)]

print('='*50)
print('IQR-BASED OUTLIER DETECTION REPORT')
print('='*50)
print(f'Q1 (25th Percentile)  : {Q1:.4f} kWh')
print(f'Q3 (75th Percentile)  : {Q3:.4f} kWh')
print(f'IQR                   : {IQR:.4f} kWh')
print(f'Lower Bound           : {lower_bound:.4f} kWh')
print(f'Upper Bound           : {upper_bound:.4f} kWh')
print(f'Total Outliers Detected: {len(outliers)}')
print(f'Outlier Percentage    : {len(outliers)/len(df_clean)*100:.2f}%')

In [ ]:
df_clean['Is_Outlier'] = ((df_clean['Consumption_KWh'] < lower_bound) |
                           (df_clean['Consumption_KWh'] > upper_bound)).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

normal  = df_clean[df_clean['Is_Outlier']==0]['Consumption_KWh']
out_pts = df_clean[df_clean['Is_Outlier']==1]['Consumption_KWh']

axes[0].scatter(range(len(normal)), normal, alpha=0.3, color='steelblue', s=8, label='Normal')
axes[0].scatter(outliers.index, out_pts, color='red', s=30, zorder=5, label=f'Outliers ({len(outliers)})')
axes[0].axhline(upper_bound, color='orange', linestyle='--', linewidth=1.5, label=f'Upper Bound: {upper_bound:.2f}')
axes[0].axhline(lower_bound, color='purple', linestyle='--', linewidth=1.5, label=f'Lower Bound: {lower_bound:.2f}')
axes[0].set_title('Outlier Detection — Scatter View')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Consumption (kWh)')
axes[0].legend(fontsize=9)

axes[1].hist(normal, bins=40, color='steelblue', edgecolor='black', alpha=0.7, label='Normal')
axes[1].hist(out_pts, bins=20, color='red', edgecolor='black', alpha=0.7, label='Outliers')
axes[1].axvline(lower_bound, color='purple', linestyle='--', linewidth=1.5)
axes[1].axvline(upper_bound, color='orange', linestyle='--', linewidth=1.5)
axes[1].set_title('Outlier Distribution Histogram')
axes[1].set_xlabel('Consumption (kWh)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('iqr_outliers.png', bbox_inches='tight')
plt.show()

In [ ]:
df_no_outliers = df_clean[df_clean['Is_Outlier'] == 0].copy()
print(f'Records before outlier removal : {len(df_clean)}')
print(f'Records after outlier removal  : {len(df_no_outliers)}')
print(f'Records removed                : {len(df_clean) - len(df_no_outliers)}')

## Section 8: GroupBy / Aggregation Analysis

In [ ]:
if 'Month' in df_clean.columns:
    monthly_stats = df_clean.groupby('Month')['Consumption_KWh'].agg(
        Total='sum',
        Mean='mean',
        Median='median',
        Std='std',
        Min='min',
        Max='max',
        Count='count'
    ).round(3)
    print('Monthly Aggregation Statistics:')
    print(monthly_stats)

In [ ]:
if 'IsWeekend' in df_clean.columns:
    day_type_stats = df_clean.groupby('Day_Type')['Consumption_KWh'].agg(
        Total='sum', Mean='mean', Median='median', Std='std', Count='count'
    ).round(3)
    print('Weekday vs Weekend Aggregation:')
    print(day_type_stats)

In [ ]:
if 'Season' in df_clean.columns:
    season_stats = df_clean.groupby('Season')['Consumption_KWh'].agg(
        Total='sum', Mean='mean', Median='median', Std='std', Count='count'
    ).round(3)
    print('Seasonal Aggregation Statistics:')
    print(season_stats)

In [ ]:
usage_group = df_clean.groupby('Usage_Category')['Consumption_KWh'].agg(
    Count='count', Mean='mean', Total='sum'
).round(3)
print('Usage Category Aggregation:')
print(usage_group)

In [ ]:
if 'Month' in df_clean.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    monthly_stats['Mean'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Average Monthly Energy Consumption')
    axes[0].set_xlabel('Month')
    axes[0].set_ylabel('Mean Consumption (kWh)')
    axes[0].tick_params(axis='x', rotation=0)

    monthly_stats['Total'].plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='black')
    axes[1].set_title('Total Monthly Energy Consumption')
    axes[1].set_xlabel('Month')
    axes[1].set_ylabel('Total Consumption (kWh)')
    axes[1].tick_params(axis='x', rotation=0)

    plt.tight_layout()
    plt.savefig('monthly_bar_charts.png', bbox_inches='tight')
    plt.show()

In [ ]:
if 'Season' in df_clean.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    season_stats['Mean'].plot(kind='bar', ax=axes[0], color=['#5DADE2','#2ECC71','#F39C12','#E74C3C'], edgecolor='black')
    axes[0].set_title('Average Consumption by Season')
    axes[0].set_xlabel('Season')
    axes[0].set_ylabel('Mean kWh')
    axes[0].tick_params(axis='x', rotation=15)

    sizes  = season_stats['Count']
    labels = season_stats.index
    axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', startangle=140,
                colors=['#5DADE2','#2ECC71','#F39C12','#E74C3C'])
    axes[1].set_title('Season-wise Record Distribution')

    plt.tight_layout()
    plt.savefig('seasonal_charts.png', bbox_inches='tight')
    plt.show()

## Section 9: Line Charts — Trend Analysis

In [ ]:
if date_col and date_col in df_clean.columns:
    ts = df_clean[[date_col, 'Consumption_KWh']].dropna().sort_values(date_col)
    ts_daily = ts.groupby(date_col)['Consumption_KWh'].mean().reset_index()

    plt.figure(figsize=(15, 5))
    plt.plot(ts_daily[date_col], ts_daily['Consumption_KWh'],
             color='steelblue', linewidth=0.8, alpha=0.7, label='Daily Average')
    rolling = ts_daily['Consumption_KWh'].rolling(window=30, min_periods=1)
    plt.plot(ts_daily[date_col], rolling.mean(),
             color='red', linewidth=2, label='30-Day Rolling Mean')
    plt.axhline(df_clean['Consumption_KWh'].mean(), color='green', linestyle='--',
                linewidth=1.5, label=f"Overall Mean: {df_clean['Consumption_KWh'].mean():.2f} kWh")
    plt.title('Energy Consumption Trend Over Time')
    plt.xlabel('Date')
    plt.ylabel('Consumption (kWh)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('consumption_trend.png', bbox_inches='tight')
    plt.show()
else:
    plt.figure(figsize=(14, 4))
    plt.plot(df_clean['Consumption_KWh'].reset_index(drop=True), color='steelblue', linewidth=0.7)
    plt.axhline(df_clean['Consumption_KWh'].mean(), color='red', linestyle='--',
                label=f"Mean: {df_clean['Consumption_KWh'].mean():.2f}")
    plt.title('Energy Consumption Over Records')
    plt.xlabel('Record Index')
    plt.ylabel('Consumption (kWh)')
    plt.legend()
    plt.tight_layout()
    plt.savefig('consumption_trend.png', bbox_inches='tight')
    plt.show()

In [ ]:
if 'Month' in df_clean.columns and 'Year' in df_clean.columns:
    years = sorted(df_clean['Year'].dropna().unique())
    if len(years) > 1:
        plt.figure(figsize=(12, 5))
        for yr in years:
            ym = df_clean[df_clean['Year']==yr].groupby('Month')['Consumption_KWh'].mean()
            plt.plot(ym.index, ym.values, marker='o', linewidth=2, label=str(yr))
        plt.title('Month-wise Consumption by Year')
        plt.xlabel('Month')
        plt.ylabel('Mean Consumption (kWh)')
        plt.xticks(range(1,13))
        plt.legend(title='Year')
        plt.tight_layout()
        plt.savefig('yearly_monthly_trend.png', bbox_inches='tight')
        plt.show()

## Section 10: Statistical Tests

### 10.1 Chi-Square Test

In [ ]:
chi2_data = df_clean.copy()
chi2_data['Outlier_Label'] = chi2_data['Is_Outlier'].map({0:'Normal', 1:'Outlier'})

if 'Season' in chi2_data.columns:
    contingency = pd.crosstab(chi2_data['Season'], chi2_data['Usage_Category'])
    group_label = 'Season'
elif 'IsWeekend' in chi2_data.columns:
    contingency = pd.crosstab(chi2_data['Day_Type'], chi2_data['Usage_Category'])
    group_label = 'Day Type'
else:
    contingency = pd.crosstab(chi2_data['Outlier_Label'], chi2_data['Usage_Category'])
    group_label = 'Outlier Status'

print('Contingency Table:')
print(contingency)

chi2_stat, p_value_chi2, dof, expected = chi2_contingency(contingency)

print('\n' + '='*55)
print('CHI-SQUARE TEST RESULT')
print('='*55)
print(f'Chi-Square Statistic : {chi2_stat:.4f}')
print(f'Degrees of Freedom   : {dof}')
print(f'P-Value              : {p_value_chi2:.6f}')
print(f'Alpha Level          : 0.05')
print('\nInterpretation:')
if p_value_chi2 < 0.05:
    print(f'  REJECT H0 — There IS a significant association between')
    print(f'  {group_label} and Usage Category (p < 0.05).')
    print(f'  Energy consumption behavior varies significantly across {group_label}.')
else:
    print(f'  FAIL TO REJECT H0 — No significant association between')
    print(f'  {group_label} and Usage Category (p >= 0.05).')

In [ ]:
plt.figure(figsize=(10, 5))
contingency_pct = contingency.div(contingency.sum(axis=1), axis=0) * 100
contingency_pct.plot(kind='bar', stacked=True, colormap='Set2', edgecolor='black', ax=plt.gca())
plt.title(f'Chi-Square Test — {group_label} vs Usage Category (Stacked %)')
plt.xlabel(group_label)
plt.ylabel('Percentage (%)')
plt.xticks(rotation=15)
plt.legend(title='Usage Category', bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.savefig('chi_square_chart.png', bbox_inches='tight')
plt.show()

### 10.2 T-Test

In [ ]:
if 'IsWeekend' in df_clean.columns:
    group_a = df_clean[df_clean['IsWeekend']==0]['Consumption_KWh'].dropna()
    group_b = df_clean[df_clean['IsWeekend']==1]['Consumption_KWh'].dropna()
    label_a, label_b = 'Weekday', 'Weekend'
elif 'Season' in df_clean.columns:
    seasons_sorted = df_clean.groupby('Season')['Consumption_KWh'].mean().sort_values(ascending=False)
    label_a = seasons_sorted.index[0]
    label_b = seasons_sorted.index[-1]
    group_a = df_clean[df_clean['Season']==label_a]['Consumption_KWh'].dropna()
    group_b = df_clean[df_clean['Season']==label_b]['Consumption_KWh'].dropna()
else:
    median_val = df_clean['Consumption_KWh'].median()
    group_a = df_clean[df_clean['Consumption_KWh'] > median_val]['Consumption_KWh']
    group_b = df_clean[df_clean['Consumption_KWh'] <= median_val]['Consumption_KWh']
    label_a, label_b = 'Above Median', 'Below Median'

t_stat, p_value_t = ttest_ind(group_a, group_b, equal_var=False)

print('='*55)
print('INDEPENDENT SAMPLES T-TEST RESULT')
print('='*55)
print(f'Group A : {label_a}  (n={len(group_a)}, Mean={group_a.mean():.4f} kWh)')
print(f'Group B : {label_b}  (n={len(group_b)}, Mean={group_b.mean():.4f} kWh)')
print(f'\nT-Statistic  : {t_stat:.4f}')
print(f'P-Value      : {p_value_t:.6f}')
print(f'Alpha Level  : 0.05')
print('\nInterpretation:')
if p_value_t < 0.05:
    print(f'  REJECT H0 — Statistically significant difference in mean')
    print(f'  consumption between {label_a} and {label_b} (p < 0.05).')
else:
    print(f'  FAIL TO REJECT H0 — No significant difference between')
    print(f'  {label_a} and {label_b} consumption (p >= 0.05).')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(group_a, bins=30, alpha=0.7, color='steelblue', edgecolor='black', label=label_a)
axes[0].hist(group_b, bins=30, alpha=0.7, color='salmon', edgecolor='black', label=label_b)
axes[0].axvline(group_a.mean(), color='blue', linestyle='--', linewidth=2)
axes[0].axvline(group_b.mean(), color='red', linestyle='--', linewidth=2)
axes[0].set_title(f'T-Test: {label_a} vs {label_b}')
axes[0].set_xlabel('Consumption (kWh)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].boxplot([group_a, group_b], patch_artist=True,
                labels=[label_a, label_b],
                boxprops=dict(facecolor='#AED6F1'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='o', color='grey', alpha=0.4))
axes[1].set_title('Comparison Box Plot')
axes[1].set_ylabel('Consumption (kWh)')
p_text = f'p = {p_value_t:.4f}' + (' *' if p_value_t < 0.05 else ' (ns)')
axes[1].text(1.5, group_a.max()*0.9, p_text, ha='center', fontsize=12,
             color='green' if p_value_t < 0.05 else 'grey')

plt.tight_layout()
plt.savefig('ttest_charts.png', bbox_inches='tight')
plt.show()

## Section 11: Anomaly / Spike Detection (Threshold-Based)

In [ ]:
mean_c = df_clean['Consumption_KWh'].mean()
std_c  = df_clean['Consumption_KWh'].std()

df_clean['Z_Score'] = (df_clean['Consumption_KWh'] - mean_c) / std_c

df_clean['Spike_Flag'] = (df_clean['Z_Score'].abs() > 2.5).astype(int)

spike_records = df_clean[df_clean['Spike_Flag'] == 1]

print('='*50)
print('CONSUMPTION SPIKE DETECTION SUMMARY')
print('='*50)
print(f'Threshold (|Z| > 2.5 sigma) applied.')
print(f'Normal readings   : {len(df_clean) - len(spike_records)}')
print(f'Spike readings    : {len(spike_records)}')
print(f'Spike Percentage  : {len(spike_records)/len(df_clean)*100:.2f}%')
print(f'\nTop 10 Highest Consumption Spikes:')
print(df_clean.nlargest(10, 'Consumption_KWh')[['Consumption_KWh','Z_Score','Spike_Flag']].to_string())

In [ ]:
if 'Month' in df_clean.columns:
    spike_by_month = df_clean.groupby('Month')['Spike_Flag'].sum().reset_index()
    spike_by_month.columns = ['Month', 'Spike_Count']

    plt.figure(figsize=(11, 4))
    plt.bar(spike_by_month['Month'], spike_by_month['Spike_Count'],
            color='tomato', edgecolor='black')
    plt.title('Monthly Consumption Spike Count')
    plt.xlabel('Month')
    plt.ylabel('Number of Spike Events')
    plt.xticks(spike_by_month['Month'])
    plt.tight_layout()
    plt.savefig('spike_bar_chart.png', bbox_inches='tight')
    plt.show()

## Section 12: User Input Section

In [ ]:
print('='*55)
print('USER INPUT — PERSONALISED ANALYSIS')
print('='*55)

# Example user consumption value
user_consumption = 25.0

print(f'Using sample consumption value: {user_consumption} kWh')

user_z = (user_consumption - mean_c) / std_c
percentile = stats.percentileofscore(df_clean['Consumption_KWh'].dropna(), user_consumption)

print(f'\nYour Consumption       : {user_consumption:.4f} kWh')
print(f'Dataset Mean           : {mean_c:.4f} kWh')
print(f'Dataset Std Dev        : {std_c:.4f} kWh')
print(f'Your Z-Score           : {user_z:.4f}')
print(f'Percentile Rank        : {percentile:.1f}th percentile')

if user_consumption < lower_bound:
    status = 'UNUSUALLY LOW — may indicate meter issue or very low usage.'
elif user_consumption > upper_bound:
    status = 'ANOMALOUS HIGH — potential spike, appliance fault, or overbilling.'
elif user_consumption > mean_c + std_c:
    status = 'ABOVE AVERAGE — slightly high consumption. Review appliance usage.'
elif user_consumption < mean_c - std_c:
    status = 'BELOW AVERAGE — efficient usage. Keep it up!'
else:
    status = 'NORMAL — within expected range.'

print(f'Status                 : {status}')

if user_consumption > mean_c:
    savings_kwh = user_consumption - mean_c
    print(f'\nPotential daily savings if reduced to average: {savings_kwh:.4f} kWh')

In [ ]:
try:
    if 'Month' in df_clean.columns:
        user_month = 6
        print(f"Using sample month: {user_month}")
        if user_month not in range(1, 13):
            raise ValueError
    else:
        raise ValueError
except ValueError:
    user_month = int(df_clean['Month'].mode()[0]) if 'Month' in df_clean.columns else None
    if user_month:
        print(f'Using most common month: {user_month}')

if user_month and 'Month' in df_clean.columns:
    month_data = df_clean[df_clean['Month'] == user_month]['Consumption_KWh'].dropna()
    month_names = {1:'January',2:'February',3:'March',4:'April',5:'May',6:'June',
                   7:'July',8:'August',9:'September',10:'October',11:'November',12:'December'}
    print(f'\n--- Analysis for {month_names.get(user_month, user_month)} ---')
    print(f'Records     : {len(month_data)}')
    print(f'Mean        : {month_data.mean():.4f} kWh')
    print(f'Median      : {month_data.median():.4f} kWh')
    print(f'Std Dev     : {month_data.std():.4f} kWh')
    print(f'Max         : {month_data.max():.4f} kWh')
    print(f'Min         : {month_data.min():.4f} kWh')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(month_data, bins=25, color='steelblue', edgecolor='black', alpha=0.85)
    axes[0].axvline(month_data.mean(), color='red', linestyle='--', linewidth=2,
                    label=f"Mean: {month_data.mean():.2f}")
    axes[0].axvline(user_consumption, color='green', linestyle='--', linewidth=2,
                    label=f"Your input: {user_consumption:.2f}")
    axes[0].set_title(f'{month_names.get(user_month,user_month)} — Consumption Histogram')
    axes[0].set_xlabel('kWh')
    axes[0].legend(fontsize=9)

    axes[1].boxplot(month_data, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'),
                    medianprops=dict(color='red', linewidth=2))
    axes[1].axhline(user_consumption, color='green', linestyle='--', linewidth=2,
                    label=f'Your input: {user_consumption:.2f}')
    axes[1].set_title(f'{month_names.get(user_month,user_month)} — Box Plot')
    axes[1].set_ylabel('kWh')
    axes[1].legend(fontsize=9)

    plt.tight_layout()
    plt.savefig(f'user_month_{user_month}_analysis.png', bbox_inches='tight')
    plt.show()

## Section 13: Final Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('Smart Energy Consumption — Analytical Dashboard', fontsize=18, fontweight='bold', y=0.98)

ax1 = fig.add_subplot(3, 3, 1)
ax1.hist(df_clean['Consumption_KWh'], bins=35, color='steelblue', edgecolor='black', alpha=0.8)
ax1.axvline(df_clean['Consumption_KWh'].mean(), color='red', linestyle='--', linewidth=2)
ax1.set_title('Consumption Distribution')
ax1.set_xlabel('kWh')
ax1.set_ylabel('Frequency')

ax2 = fig.add_subplot(3, 3, 2)
ax2.boxplot(df_clean['Consumption_KWh'].dropna(), patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
            medianprops=dict(color='red', linewidth=2))
ax2.set_title('Overall Box Plot')
ax2.set_ylabel('kWh')

ax3 = fig.add_subplot(3, 3, 3)
cat_counts = df_clean['Usage_Category'].value_counts()
colors_bar = ['#2ECC71','#F39C12','#E74C3C']
ax3.bar(cat_counts.index, cat_counts.values, color=colors_bar[:len(cat_counts)], edgecolor='black')
ax3.set_title('Usage Category Distribution')
ax3.set_xlabel('Category')
ax3.set_ylabel('Count')

ax4 = fig.add_subplot(3, 3, 4)
if 'Month' in df_clean.columns:
    m_mean = df_clean.groupby('Month')['Consumption_KWh'].mean()
    ax4.plot(m_mean.index, m_mean.values, marker='o', color='darkorange', linewidth=2)
    ax4.set_title('Monthly Avg Consumption')
    ax4.set_xlabel('Month')
    ax4.set_ylabel('Mean kWh')
else:
    ax4.plot(df_clean['Consumption_KWh'].reset_index(drop=True), color='darkorange', linewidth=0.5)
    ax4.set_title('Consumption Over Records')

ax5 = fig.add_subplot(3, 3, 5)
spike_label = ['Normal', 'Spike']
spike_counts = df_clean['Spike_Flag'].value_counts().sort_index()
ax5.pie(spike_counts, labels=spike_label, autopct='%1.1f%%',
        colors=['#5DADE2','#E74C3C'], startangle=90)
ax5.set_title('Normal vs Spike Records')

ax6 = fig.add_subplot(3, 3, 6)
outlier_counts = df_clean['Is_Outlier'].value_counts().sort_index()
ax6.pie(outlier_counts, labels=['Normal','Outlier'], autopct='%1.1f%%',
        colors=['#82E0AA','#F1948A'], startangle=90)
ax6.set_title('IQR Outlier Proportion')

ax7 = fig.add_subplot(3, 3, 7)
if 'Season' in df_clean.columns:
    s_mean = df_clean.groupby('Season')['Consumption_KWh'].mean().sort_values(ascending=False)
    ax7.barh(s_mean.index, s_mean.values, color='#85C1E9', edgecolor='black')
    ax7.set_title('Avg Consumption by Season')
    ax7.set_xlabel('Mean kWh')
elif 'DayOfWeek' in df_clean.columns:
    d_mean = df_clean.groupby('DayOfWeek')['Consumption_KWh'].mean()
    days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
    ax7.bar([days[i] for i in d_mean.index], d_mean.values, color='#85C1E9', edgecolor='black')
    ax7.set_title('Avg Consumption by Day of Week')
    ax7.set_ylabel('Mean kWh')
else:
    ax7.hist(df_clean['Consumption_Standardized'], bins=30, color='#85C1E9', edgecolor='black')
    ax7.set_title('Standardized Consumption')

ax8 = fig.add_subplot(3, 3, 8)
if 'Month' in df_clean.columns:
    spike_m = df_clean.groupby('Month')['Spike_Flag'].sum()
    ax8.bar(spike_m.index, spike_m.values, color='tomato', edgecolor='black')
    ax8.set_title('Spikes per Month')
    ax8.set_xlabel('Month')
    ax8.set_ylabel('Spike Count')
else:
    ax8.hist(df_clean['Z_Score'], bins=35, color='tomato', edgecolor='black', alpha=0.8)
    ax8.set_title('Z-Score Distribution')

ax9 = fig.add_subplot(3, 3, 9)
if 'Month' in df_clean.columns:
    m_total = df_clean.groupby('Month')['Consumption_KWh'].sum()
    ax9.fill_between(m_total.index, m_total.values, color='#A9CCE3', alpha=0.8)
    ax9.plot(m_total.index, m_total.values, color='navy', marker='o', linewidth=2)
    ax9.set_title('Total Monthly Consumption')
    ax9.set_xlabel('Month')
    ax9.set_ylabel('Total kWh')
else:
    ax9.hist(df_clean['Consumption_Log'], bins=35, color='#A9CCE3', edgecolor='black', alpha=0.8)
    ax9.set_title('Log Consumption Distribution')

plt.tight_layout()
plt.savefig('analytical_dashboard.png', bbox_inches='tight', dpi=120)
plt.show()

## Section 14: Final Insights and Conclusion

In [ ]:
print('='*65)
print('     SMART ENERGY CONSUMPTION ANALYSIS — FINAL INSIGHTS')
print('='*65)

print('\n[1] DATASET OVERVIEW')
print(f'    Total records analysed     : {len(df_clean):,}')
print(f'    Numeric features           : {len(numeric_cols)}')
print(f'    Energy column used         : {energy_col}')

print('\n[2] DESCRIPTIVE STATISTICS')
print(f'    Mean Consumption           : {df_clean["Consumption_KWh"].mean():.4f} kWh')
print(f'    Median Consumption         : {df_clean["Consumption_KWh"].median():.4f} kWh')
print(f'    Std Deviation              : {df_clean["Consumption_KWh"].std():.4f} kWh')
print(f'    Skewness                   : {df_clean["Consumption_KWh"].skew():.4f}')
if df_clean['Consumption_KWh'].skew() > 0.5:
    print('    Interpretation: Right-skewed distribution — majority of consumers')
    print('    use moderate energy; a few high-usage outliers drive the mean up.')

print('\n[3] IQR OUTLIER ANALYSIS')
print(f'    IQR Bounds                 : [{lower_bound:.4f}, {upper_bound:.4f}] kWh')
outlier_pct = len(df_clean[df_clean['Is_Outlier']==1])/len(df_clean)*100
print(f'    Outlier Records            : {len(df_clean[df_clean["Is_Outlier"]==1])} ({outlier_pct:.2f}%)')
print(f'    Insight: {outlier_pct:.1f}% of records exceed normal IQR bounds — these')
print(f'    represent potential billing anomalies or appliance faults.')

print('\n[4] SPIKE / ANOMALY DETECTION')
spike_pct = df_clean['Spike_Flag'].mean() * 100
print(f'    Spike Records (|Z|>2.5)    : {df_clean["Spike_Flag"].sum()} ({spike_pct:.2f}%)')
if spike_pct > 5:
    print('    High spike frequency detected. Suggests irregular usage patterns')
    print('    possibly linked to high-load appliances or seasonal demand.')
else:
    print('    Spike frequency is within acceptable range (<5% of records).')

print('\n[5] CHI-SQUARE TEST')
print(f'    Chi² Statistic  : {chi2_stat:.4f}')
print(f'    P-Value         : {p_value_chi2:.6f}')
if p_value_chi2 < 0.05:
    print(f'    Result: SIGNIFICANT — Usage categories differ across {group_label}.')
    print(f'    Targeted energy-saving programs should consider {group_label} patterns.')
else:
    print(f'    Result: NOT SIGNIFICANT — No strong categorical dependency found.')

print('\n[6] T-TEST')
print(f'    T-Statistic     : {t_stat:.4f}')
print(f'    P-Value         : {p_value_t:.6f}')
if p_value_t < 0.05:
    print(f'    Result: SIGNIFICANT — {label_a} and {label_b} have meaningfully')
    print(f'    different consumption levels. Different policies needed for each group.')
else:
    print(f'    Result: NOT SIGNIFICANT — {label_a} and {label_b} consume similarly.')

if 'Month' in df_clean.columns:
    print('\n[7] SEASONAL / MONTHLY PATTERNS')
    peak_month = df_clean.groupby('Month')['Consumption_KWh'].mean().idxmax()
    low_month  = df_clean.groupby('Month')['Consumption_KWh'].mean().idxmin()
    month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                   7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    print(f'    Peak Consumption Month  : {month_names.get(peak_month, peak_month)}')
    print(f'    Lowest Consumption Month: {month_names.get(low_month, low_month)}')
    print('    Recommendation: Launch demand-management programs before peak months.')

print('\n[8] USAGE CATEGORY BREAKDOWN')
for cat in df_clean['Usage_Category'].cat.categories:
    count = len(df_clean[df_clean['Usage_Category']==cat])
    pct   = count / len(df_clean) * 100
    print(f'    {cat:<8}: {count:>6} records ({pct:>5.1f}%)')

print('\n[9] BUSINESS RECOMMENDATIONS')
print('    1. Target top 5% high-consumption households with efficiency audits.')
print('    2. Deploy early-warning alerts for daily readings exceeding Z-score 2.5.')
print('    3. Introduce time-of-use pricing to reduce peak-hour demand.')
print('    4. Offer personalised consumption dashboards per household.')
print('    5. Use IQR flagging monthly to identify potential meter tampering.')
print('    6. Investigate seasonal spikes to advise targeted conservation programs.')

print('\n[10] CONCLUSION')
print('    This analysis of smart meter data using Fundamentals of Data Analytics')
print('    techniques — EDA, IQR outlier detection, statistical hypothesis tests')
print('    (Chi-Square, T-Test), GroupBy aggregations, and trend analysis —')
print('    reveals meaningful consumption patterns across time periods.')
print('    Data-driven anomaly detection and behavioural segmentation provide')
print('    actionable pathways to reduce energy wastage, prevent overbilling,')
print('    and promote sustainable smart energy management.')
print('='*65)